### Notebook for visualizing the embeddings
- need result folder from previous notebook
- dataframe from previous notebook

In [38]:
import pandas as pd
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
import umap
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.palettes import Category10



In [ ]:
working_dir = Path("./results")

assert working_dir.is_dir, f"Working directory {working_dir} does not exist."


In [6]:
data_info = pd.read_csv(working_dir / "data_codon_bert.csv")

In [7]:
data_info

,sequence,y,len,token_len
0,AUGUUGAAAUCCCCCCUGUUCUGGAAAAUGACCACCCUGUUUGGUG...,0,1353,285
1,AUGAAACUUUUUCGUAUCCUCGAUCCUUUCACCUUAACCCUGAUCA...,0,999,214
2,AUGAGUUCUUUAAGUCAGGCUGCGAGCAGUGUGGAAAAACGCACAA...,0,1353,302
3,AUGCGUAAGUUCAUUUUCGUAUUGCUGACACUGCUUUUGGUCAGCC...,0,891,192
4,AUGUCACUAACCAGACGGAGGUUUACACAGAUUCUUGCGUCGACGU...,0,1023,223
...,...,...,...,...
2467,AUGAUUUCUCUGCUCCGGCCCCCCAAGCUCAAGUGGCUGUGGUUUU...,4,1182,248
2468,AUGCUGUUUCUGCAACUUCCAUUGCUAGCUGUGUUCCUUCCAGGUG...,4,720,152
2469,AUGGCAAUGAUCUCAGGGCUCAGUGGCAGGAAAUCCUCAACAGGGU...,4,468,96
2470,AUGGCCGCCAUCCGCAAGAAGCUGGUGAUCGUGGGGGACGGGGCCU...,4,582,115


In [73]:
from typing import Literal, Dict, List
import torch

class VizData:
    def __init__(
        self,
        working_dir: str | Path = working_dir,
        info_df: pd.DataFrame = data_info,
        choice_embedding_data: Literal["with_eos", "no_eos"] = "no_eos",
        choice_for_embedding: Literal["mean", "max", "eos"] = "mean", 

        label_info: Dict = {'ecoli':0,'marsupials':1,'monotremes':2,'human_virus':3,'more_placentals':4}

    ):

        assert working_dir.is_dir(), f"Working directory {working_dir} does not exist."

        self.info = info_df
        self.labels = info_df["y"].values
        self.label_info = label_info
        self.reverse_label_info = {v: k for k, v in label_info.items()}



        self.choice_embedding_data = choice_embedding_data
        self.choice_for_embedding = choice_for_embedding

        self.umap_df = pd.DataFrame()
        self.umap_df["labels"] = self.labels

        # use bokeh color palette
        # self.palette = Category10[10]
        # choose 5 colors for the 5 classes

        # use user def colors
        
        colors = ['blue', 'orange', 'green', 'red', 'purple']
        self.umap_df['color'] = self.umap_df['labels'].map(lambda x: colors[x])
        self.umap_df['legend'] = self.umap_df['labels'].map(self.reverse_label_info)

        self.embedding_data = self.get_embedding_data(choice_embedding_data)
        self.embeddings = self.get_embedding(choice_for_embedding, self.embedding_data)

        print(f"Loaded {self.choice_embedding_data} embedding with shape {self.embeddings.shape}.")

    @staticmethod
    def get_embedding_data(choice):
        match choice:
            case "with_eos":
                embedding_data = torch.load(working_dir / "emb_with_eos.pt")
            case "no_eos":
                # embedding_data = torch.load(working_dir / "emb_no_eos.pt") 
                embedding_data = torch.load("state_embeddings_no_eos.pt").to("cpu") 

            case _:
                raise ValueError("Invalid choice for embedding data. Choose 'with_eos' or 'no_eos'.")
            
        return embedding_data 

    @staticmethod
    def get_embedding(choice, embedding_data):
        match choice:
            case "mean":
                embeddings = embedding_data.mean(dim=1)
            case "max":
                embeddings = embedding_data.max(dim=1) 

            case _:


                raise ValueError("Invalid choice for embedding. Choose 'mean', 'max', or 'eos'.")
        return embeddings
    

In [74]:
viz = VizData(
    working_dir=working_dir,
    info_df=data_info,
    choice_embedding_data="no_eos",
    choice_for_embedding="mean",
)
 
    

Loaded no_eos embedding with shape torch.Size([2472, 768]).


In [75]:
viz.embeddings

tensor([[ 1.3910,  0.0527,  0.7102,  ..., -0.1896, -0.1321,  1.0913],
        [ 1.1524, -0.8263,  1.4642,  ..., -0.4328,  0.3754,  1.1052],
        [ 0.8135,  0.4310,  1.2375,  ..., -0.2765,  0.0313,  0.7883],
        ...,
        [-0.3719,  1.3158, -0.1802,  ..., -0.5747,  0.6163,  0.7298],
        [ 0.9579, -0.2684, -0.8703,  ...,  1.3084, -0.3091,  0.1039],
        [ 0.8417,  0.1866,  0.3847,  ...,  0.0703,  0.1270,  1.2626]])

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
import umap
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.palettes import Category10

from sklearn.preprocessing import StandardScaler, MinMaxScaler
output_notebook()


# Step 1: Normalize
scaler = StandardScaler()
# scaler = MinMaxScaler()

normalized_embeddings = scaler.fit_transform(viz.embeddings)

# # Step 3: PCA
pca = PCA(n_components=500)
pca_embeddings = pca.fit_transform(viz.embeddings)

# Step 4: UMAP
umap_model = umap.UMAP(
    n_neighbors=15,
    n_epochs=2000,
    min_dist=0.8,
    init='spectral',
    # n_neighbors=15,
    # n_neighbors=5,
    # n_neighbors=10,
    # n_neighbors=20,
    metric='euclidean',
    # n_components=2,
    random_state=42,
    # spread=1.5
)

# umap_embeddings = umap_model.fit_transform(pca_embeddings)
# umap_embeddings = umap_model.fit_transform(normalized_embeddings)
umap_embeddings = umap_model.fit_transform(viz.embeddings)

umap_df = viz.umap_df

# add umap embeddings to the dataframe
umap_df["x"] = umap_embeddings[:, 0]
umap_df["y"] = umap_embeddings[:, 1]
umap_df

# Map colors to labels


# show only human virus and ecoli 
umap_df = umap_df[umap_df['labels'].isin([0, 2, 3])]



# !!! AFTER HAVEING umap_df visualization library is your choice


# Step 5: Create Bokeh plot with tooltips

source = ColumnDataSource(umap_df)

hover = HoverTool(tooltips=[
    # ("Index", "@index"),
    # ("Label", "@label"),
    ("Label", "@legend"),
    # ("x", "@x{0.2f}"),
    # ("y", "@y{0.2f}")
])

plot = figure(
    title="UMAP for RNA CodonBERT Embeddings",
    tools=["pan,wheel_zoom,reset", hover],
    width=700, height=600
)

plot.scatter(
    'x', 'y', 
    source=source, 
    size=5, 
    color='color',
    legend_field='legend',
    alpha=0.4,
)

show(plot)


Loading BokehJS ...

/fsx/home/hermee/anaconda3/envs/ai/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/fsx/home/hermee/anaconda3/envs/ai/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
